# inspecting datasets

In [1]:
from src.extract import extract, dataframes
import pandas as pd

In [2]:
for key, value in dataframes.items():
        # Investigating missing values

        # Getting datasets that has the null values
        # getting total column values for calculating total rows and columns
        column_with_nulls = value.columns[value.isna().any()].tolist()
        if column_with_nulls:
            print(key) # dataset name
            # Getting columns that has null values
            for column_name in column_with_nulls:
                # Calculating percentage for null values
                null_counts = value[column_name].isna().sum()
                total_count = value.shape[0]
                null_value_percentage = (null_counts / total_count) * 100
                print(
                    f"{column_name}: {null_counts} missing values "
                    f"({null_value_percentage:.2f}%)"
                )

            print() # for new line

olist_orders_dataset
order_approved_at: 160 missing values (0.16%)
order_delivered_carrier_date: 1783 missing values (1.79%)
order_delivered_customer_date: 2965 missing values (2.98%)

olist_order_reviews_dataset
review_comment_title: 87656 missing values (88.34%)
review_comment_message: 58247 missing values (58.70%)

olist_products_dataset
product_category_name: 610 missing values (1.85%)
product_name_lenght: 610 missing values (1.85%)
product_description_lenght: 610 missing values (1.85%)
product_photos_qty: 610 missing values (1.85%)
product_weight_g: 2 missing values (0.01%)
product_length_cm: 2 missing values (0.01%)
product_height_cm: 2 missing values (0.01%)
product_width_cm: 2 missing values (0.01%)



In [ ]:
 # Investigating orders dataset for order_delivered_customer_date, by checking orders status column and other things
orders_df = dataframes['olist_orders_dataset']

null_rows = orders_df[orders_df['order_delivered_customer_date'].isna()]

filtered_null_rows = null_rows.loc[
    :, 
        [
            
            'order_id', 
            'customer_id', 
            'order_status', 
            'order_approved_at',
            'order_purchase_timestamp', 
            'order_delivered_carrier_date', 'order_delivered_customer_date'
            
        ]
    ]

In [ ]:
# print(filtered_null_rows['order_status'].value_counts()) # check the order status wise counts

print(filtered_null_rows[filtered_null_rows['order_status'] == 'delivered'])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_delivered_carrier_date'].isna())])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_approved_at'].isna())])

# Task 06 - Investigate Product Information

In [ ]:
product_df = dataframes['olist_products_dataset']

In [ ]:
product_df

In [ ]:
filtered_products_df = product_df.loc[:, [
    'product_category_name', 
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]]

In [ ]:
display(filtered_products_df.head())

In [ ]:
missing_product_category_name = filtered_products_df[filtered_products_df['product_category_name'].isna()]

In [ ]:
missing_product_category_name.isna().sum().any()

In [ ]:
missing_products = filtered_products_df[filtered_products_df['product_category_name'].isna() & filtered_products_df['product_name_lenght'].isna() & filtered_products_df['product_description_lenght'].isna() & filtered_products_df['product_photos_qty'].isna()]

In [ ]:
missing_products

In [ ]:
product_df[product_df['product_id'].isna()]

In [ ]:
missing_dimensions = product_df[
    product_df[
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ].isna().any(axis=1)
]

In [ ]:
missing_dimensions

In [ ]:
reviews_df = dataframes['olist_order_reviews_dataset']

In [ ]:
reviews_df.columns.tolist()

In [ ]:
reviews_df.shape

In [ ]:
# checking duplicates
reviews_df[reviews_df.duplicated()]

In [ ]:
# Checking missing records
reviews_df['review_score'].isna().sum()

In [ ]:
# checking unique 
reviews_df['review_score'].unique()

In [ ]:
# Scorewise occurances
reviews_df['review_score'].value_counts()

In [ ]:
reviews_df['review_id'].duplicated().sum()

In [ ]:
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Missing identifiers
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Duplicate identifiers
reviews_df[['review_id', 'order_id']].duplicated().sum()

In [ ]:
# Check review_id uniqueness
reviews_df['review_id'].nunique()

In [ ]:
# Review score distribution
reviews_df['review_score'].value_counts().sort_index()

In [ ]:
duplicate_reviews = reviews_df[
    reviews_df['review_id'].duplicated(keep=False)
]

duplicate_reviews.sort_values('review_id')

In [ ]:
duplicate_reviews['review_id'].value_counts().head(30)

In [ ]:
sample_review_id = duplicate_reviews['review_id'].value_counts().index[0]

duplicate_reviews[
    duplicate_reviews['review_id'] == sample_review_id
].T

In [ ]:
duplicate_reviews.groupby('review_id')['order_id'].nunique().value_counts()

In [ ]:
duplicate_reviews.groupby('review_id')['review_score'].nunique().value_counts()

# Transformation starts

- [x] Identify timestamp columns.
- [x] Understand their current Pandas data types.
- [x] Decide which columns need conversion.
- [x] Implement the conversion.
- [x] Verify the result.

In [3]:
orders_df = dataframes['olist_orders_dataset']

In [19]:
orders_df.dtypes

order_id                                    str
customer_id                                 str
order_status                           category
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [18]:
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [8]:
orders_df['order_status'].unique().tolist()

['delivered',
 'invoiced',
 'shipped',
 'processing',
 'unavailable',
 'canceled',
 'created',
 'approved']

In [9]:
orders_df['order_status'] = orders_df['order_status'].astype('category')

In [10]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [13]:
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [14]:
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [15]:
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [16]:
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [17]:
orders_df[orders_df['order_purchase_timestamp'] == 'NaT']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


# Task 08 - Investigate order timestamp consistency

### Part 1 — Investigate chronological consistency
- [x] Approval timestamp is earlier than purchase timestamp.
- [x] Carrier delivery timestamp is earlier than approval timestamp.
- [x] Customer delivery timestamp is earlier than carrier delivery timestamp.
- [x] Customer delivery timestamp is earlier than purchase timestamp.

In [41]:
orders_df[orders_df['order_approved_at'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [42]:
orders_df[orders_df['order_delivered_carrier_date'] < orders_df['order_approved_at']].shape

(1359, 8)

In [43]:
orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_delivered_carrier_date']].shape

(23, 8)

In [44]:
orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [ ]:
orders_df.head()

### Part 2 — Investigate estimated delivery
- [x] Estimated delivery date is earlier than purchase date.
- [x] Actual customer delivery is later than the estimated delivery date.

In [45]:
orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [48]:
orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_delivered_customer_date']].shape

(7827, 8)

### Part 3 — Report your findings
- [x] Number of affected records.
- [ ] A few sample records (order_id and relevant timestamps).

In [47]:
orders_df[['order_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15
